In [0]:
-- ============================================================
-- GOLD LAYER
-- Creating county-level analytics table
-- ============================================================

CREATE OR REPLACE TABLE emissions.gold.county_metrics
USING DELTA
AS

SELECT
    county_id,
    county_name,
    county_state_name,
    state_id,
    state_abbr,
    latitude,
    longitude,
    population,
    ghg_emissions_mtons_co2e,

    CASE
        WHEN population > 0
        THEN ghg_emissions_mtons_co2e / population
        ELSE NULL
    END AS emissions_per_person

FROM emissions.silver.emissions_clean;

---------------------------------------------------
SELECT *
FROM emissions.gold.county_metrics
LIMIT 10;

--------------------------------------------------
SELECT COUNT(*) AS county_gold_rows
FROM emissions.gold.county_metrics;
-------------------------------------------
--highest emissions-per-person counties
SELECT
    county_state_name,
    population,
    ghg_emissions_mtons_co2e,
    emissions_per_person
FROM emissions.gold.county_metrics
ORDER BY emissions_per_person DESC
LIMIT 10;

----------------------------------------

-- GOLD LAYER
-- Creating state-level emissions aggregates


CREATE OR REPLACE TABLE emissions.gold.state_emissions
USING DELTA
AS

WITH state_totals AS (

    SELECT
        state_abbr,
        SUM(population) AS total_population,
        SUM(ghg_emissions_mtons_co2e) AS total_emissions

    FROM emissions.silver.emissions_clean

    GROUP BY state_abbr
)

SELECT
    state_abbr,
    total_population,
    total_emissions,

    total_emissions / NULLIF(total_population, 0)
        AS emissions_per_person,

    100.0 * total_emissions
        / SUM(total_emissions) OVER ()
        AS percentage_of_us_emissions,

    DENSE_RANK() OVER (
        ORDER BY total_emissions DESC
    ) AS emissions_rank

FROM state_totals;

-------------------------------------------------------------

SELECT *
FROM emissions.gold.state_emissions
ORDER BY emissions_rank;

-------------------------------------

SELECT *
FROM emissions.gold.state_emissions
WHERE emissions_rank <= 10
ORDER BY emissions_rank;

---------------------------------
-- GOLD LAYER
-- Creating summary metrics for dashboard reporting


CREATE OR REPLACE TABLE emissions.gold.emissions_summary
USING DELTA
AS

SELECT

    SUM(total_emissions) AS total_us_emissions,

    SUM(
        CASE
            WHEN emissions_rank <= 10
            THEN total_emissions
            ELSE 0
        END
    ) AS top10_emissions,

    100.0 *
    SUM(
        CASE
            WHEN emissions_rank <= 10
            THEN total_emissions
            ELSE 0
        END
    )
    / SUM(total_emissions)
    AS top10_percentage

FROM emissions.gold.state_emissions;

-------------------------------------------
SELECT *
FROM emissions.gold.emissions_summary;
--------------------------------------------


SELECT COUNT(*) FROM emissions.bronze.emissions_raw;

SELECT COUNT(*) FROM emissions.silver.emissions_clean;

SELECT COUNT(*) FROM emissions.gold.county_metrics;

SELECT * FROM emissions.gold.state_emissions
ORDER BY emissions_rank
LIMIT 10;

SELECT * FROM emissions.gold.emissions_summary;